# cAPTure: fold-local model preprocessing audit

This CPU-only notebook fits and audits the reviewed 103-column primary packet representation for both development folds. Numeric parameters and variance masks are fitted only on each fold's training scenarios. Validation packets are transformed without refitting. The notebook saves compact JSON preprocessors and reports to Drive; it does not materialize transformed packet tables, build windows, or train a model.


## 1. Mount Drive and load the project


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_preprocess.py",
    PROJECT_ROOT / "code/python/tests/test_capture_preprocess.py",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
    PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml",
    PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("CPU preprocessing-audit environment is ready.")


Mounted at /content/drive
CPU preprocessing-audit environment is ready.


## 2. Run synthetic contract checks


In [2]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for test_pattern in ("test_capture_feature_profile.py", "test_capture_preprocess.py"):
    subprocess.run(
        [sys.executable, "-m", "unittest", "discover",
         "-s", str(PROJECT_ROOT / "code/python/tests"),
         "-p", test_pattern, "-v"],
        env=test_environment,
        cwd=PROJECT_ROOT,
        check=True,
    )
print("Synthetic preprocessing checks passed.")


Synthetic preprocessing checks passed.


## 3. Configure the FULL_DEV audit

The completed canonical preparation run is immutable input. A batch size of 100,000 keeps peak CPU memory bounded while the two fold-specific preprocessors are fitted and verified.


In [3]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from utils.capture_preprocess import run_capture_preprocessing_audit

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs" / "20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 100_000

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_preprocessing"
DRIVE_RUN_DIR = DRIVE_ROOT / "preprocessing_runs" / RUN_ID

print(f"Prepared input: {PREPARED_RUN_DIR}")
print(f"Audit output: {DRIVE_RUN_DIR}")


Prepared input: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Audit output: /content/drive/MyDrive/capture_gate0/preprocessing_runs/20260918T233005_532313Z_preprocessing


## 4. Fit, transform, and persist the audit

This is the long-running section. Each fold first scans only its training scenarios to fit numeric statistics and the variance mask. It then transforms every development scenario by batches to verify row conservation, fixed width, and finite values. No transformed packet batch is retained.


In [4]:
AUDIT = run_capture_preprocessing_audit(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=DRIVE_RUN_DIR,
    batch_size=BATCH_SIZE,
)
print(f"Saved preprocessing audit: {DRIVE_RUN_DIR}")


Fitting preprocessing fold A...
Auditing fold A train scenario train_empty_conn...
Auditing fold A train scenario train_qos_mid...
Auditing fold A validation scenario train_dollar_char...
Auditing fold A validation scenario train_slash_char...
Auditing fold A validation scenario train_sub_exf...
Fitting preprocessing fold B...
Auditing fold B validation scenario train_empty_conn...
Auditing fold B validation scenario train_qos_mid...
Auditing fold B train scenario train_dollar_char...
Auditing fold B train scenario train_slash_char...
Auditing fold B train scenario train_sub_exf...
Saved preprocessing audit: /content/drive/MyDrive/capture_gate0/preprocessing_runs/20260918T233005_532313Z_preprocessing


## 5. Review the fixed representation and fold masks


In [5]:
display(pd.DataFrame({
    "position": range(AUDIT["model_feature_count"]),
    "feature": AUDIT["model_feature_names"],
}))

fold_rows = []
masked_rows = []
numeric_rows = []
for fold, fold_report in AUDIT["folds"].items():
    fold_rows.append({
        "fold": fold,
        "training_scenarios": fold_report["train_scenarios"],
        "validation_scenarios": fold_report["validation_scenarios"],
        "feature_count": fold_report["feature_count"],
        "active_feature_count": fold_report["active_feature_count"],
        "masked_feature_count": fold_report["masked_feature_count"],
    })
    for feature in fold_report["masked_features"]:
        masked_rows.append({"fold": fold, "masked_feature": feature})
    artifact = json.loads(
        (DRIVE_RUN_DIR / fold_report["preprocessor_artifact"]).read_text(encoding="utf-8")
    )
    for feature, parameters in artifact["numeric_parameters"].items():
        numeric_rows.append({"fold": fold, "feature": feature, **parameters})

print("Fold summary")
display(pd.DataFrame(fold_rows))
print("Fold-training masked features")
display(pd.DataFrame(masked_rows))
print("Fold-training numeric parameters")
display(pd.DataFrame(numeric_rows))


,position,feature
0,0,destination_is_broadcast
1,1,destination_is_multicast
2,2,is_arp
3,3,is_ipv4
4,4,is_ipv6
...,...,...
98,98,mqtt_subscription_qos_1
99,99,mqtt_subscription_qos_2
100,100,mqtt_subscription_qos_3
101,101,ssh_direction_0


Fold summary


,fold,training_scenarios,validation_scenarios,feature_count,active_feature_count,masked_feature_count
0,A,"[train_empty_conn, train_qos_mid]","[train_dollar_char, train_slash_char, train_su...",103,89,14
1,B,"[train_dollar_char, train_slash_char, train_su...","[train_empty_conn, train_qos_mid]",103,92,11


Fold-training masked features


,fold,masked_feature
0,A,tcp_flag_cwr
1,A,tcp_flag_ece
2,A,tcp_flag_urg
3,A,tcp_source_port_role_zero_or_reserved
4,A,tcp_destination_port_role_zero_or_reserved
5,A,ipv4_dscp_bit_0
6,A,ipv4_dscp_bit_2
7,A,ipv4_dscp_bit_4
8,A,ipv4_dscp_bit_5
9,A,mqtt_qos_3


Fold-training numeric parameters


,fold,feature,transform,training_nonnull,training_mean,training_standard_deviation,scaling_divisor
0,A,frame_length,log1p_standardize,2675496,4.507300,0.670508,0.670508
1,A,ipv4_fragment_offset,standardize,2525968,0.001147,0.043713,0.043713
2,A,ipv4_length,log1p_standardize,2525968,4.206060,0.714507,0.714507
3,A,ipv4_ttl,standardize,2525968,61.843113,5.982249,5.982249
4,A,ssh_padding_length,standardize,11732,8.429850,2.026993,2.026993
5,A,tcp_header_length,standardize,2524036,29.451421,4.746904,4.746904
6,A,tcp_payload_length,log1p_standardize,2524036,1.448965,1.936554,1.936554
7,A,tcp_window_value,log1p_standardize,2524036,5.579328,2.295106,2.295106
8,B,frame_length,log1p_standardize,11533878,4.430576,0.606745,0.606745
9,B,ipv4_fragment_offset,standardize,11193055,0.001182,0.044369,0.044369


## 6. Review scenario-level transformation checks


In [6]:
scenario_rows = []
for fold, fold_report in AUDIT["folds"].items():
    for scenario, scenario_audit in fold_report["scenario_audits"].items():
        scenario_rows.append({"fold": fold, "scenario": scenario, **scenario_audit})
scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table)

assert AUDIT["model_feature_count"] == 103
assert AUDIT["transformed_packet_artifacts_written"] is False
assert scenario_table["feature_count"].eq(103).all()
assert scenario_table["finite_values"].all()
assert all(
    len(report["train_scenarios"]) + len(report["validation_scenarios"]) == 5
    for report in AUDIT["folds"].values()
)
print("The fold-local preprocessing audit completed without structural errors.")
print("Review the masks and numeric parameters before freezing the contract.")


,fold,scenario,rows,feature_count,finite_values,nonzero_values,maximum_absolute_value,active_features_zero_in_this_scenario,partition
0,A,train_empty_conn,1175779,103,True,16937023,45.726788,[ipv4_dscp_bit_1],train
1,A,train_qos_mid,1499717,103,True,21422239,45.726788,"[mqtt_message_type_10, mqtt_message_type_11, m...",train
2,A,train_dollar_char,2882555,103,True,43659826,45.726788,"[mqtt_message_type_10, mqtt_message_type_11, m...",validation
3,A,train_slash_char,6425516,103,True,88354700,45.726788,[ipv4_dscp_bit_1],validation
4,A,train_sub_exf,2225807,103,True,33211047,45.726788,[mqtt_reserved_flag_4],validation
5,B,train_empty_conn,1175779,103,True,16937023,45.050262,"[ipv4_dscp_bit_1, mqtt_qos_3, mqtt_reserved_fl...",validation
6,B,train_qos_mid,1499717,103,True,21422239,45.050262,"[mqtt_message_type_10, mqtt_message_type_11, m...",validation
7,B,train_dollar_char,2882555,103,True,43659826,45.050262,"[mqtt_message_type_10, mqtt_message_type_11, m...",train
8,B,train_slash_char,6425516,103,True,88354705,45.050262,"[ipv4_dscp_bit_1, mqtt_qos_3]",train
9,B,train_sub_exf,2225807,103,True,33211059,45.050262,[mqtt_reserved_flag_4],train


The fold-local preprocessing audit completed without structural errors.
Review the masks and numeric parameters before freezing the contract.
